# Week 02 — Deep CVR — factorization machines, Wide & Deep, DLRM

**Goal.** Implement the embedding-based lineage of production ads models and find out, with numbers, whether it beats trees at this scale.

**Deliverable.** FM + a DLRM-lite in PyTorch, a comparison table against Week 1's LightGBM, and a one-page verdict.

**Rough shape of the week.** 2h reading (DLRM, Wide & Deep) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The question this week actually answers

Not "can I implement DLRM" — you can. The question is **at what point do embeddings plus
an MLP beat gradient-boosted trees on tabular ads data, and why**. The honest published
answer (see `papers/shwartzziv2021-tabular-dl-is-not-all-you-need.pdf`) is: often they
don't, until the categorical cardinality and the row count are both large.

So the deliverable is a *curve*, not a point. Train each model at 1%, 10%, and 100% of
the rows and plot AUC against training-set size. Where the lines cross — if they cross —
is the finding.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
print(f"{len(df):,} rows, {df.timestamp.max()/86400:.1f} days")

sp = split.time_split(df, "timestamp", train_frac=0.7, val_frac=0.1)
split.check_no_leakage(df, sp)
print(sp)

train, val, test = sp.apply(df)

## 1. Embedding tables

Map each `cat*` column to a contiguous index range, then to a learned vector. Two
decisions to make deliberately and record:

- **Vocabulary**: hash to a fixed size (like Week 1) or build an index of values seen in
  train? Hashing collides; indexing has to handle unseen values at test time. Ads systems
  hash. Try both and measure the difference.
- **Embedding dim**: 16 is a reasonable default for everything. Dimension-per-feature
  proportional to `log(cardinality)` is what large systems actually do.

In [ ]:
import torch
import torch.nn as nn

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

class EmbeddingBag(nn.Module):
    """One embedding table per categorical column, concatenated."""
    def __init__(self, cardinalities, dim=16):
        super().__init__()
        # TODO
        raise NotImplementedError

## 2. Factorization machine

FM = linear terms + all pairwise interactions, computed in O(nd) with the classic
identity

$$\sum_{i<j}\langle v_i, v_j\rangle x_i x_j = \tfrac12\sum_f\Big[\big(\sum_i v_{i,f}x_i\big)^2 - \sum_i v_{i,f}^2x_i^2\Big]$$

Implement it with that identity, not with a double loop — the point of FM is that the
interaction term is linear-time, and writing the naive version teaches you nothing except
that it is slow.

In [ ]:
class FM(nn.Module):
    def __init__(self, cardinalities, dim=16):
        super().__init__()
        # TODO: linear term + the O(nd) pairwise identity above
        raise NotImplementedError

## 3. Wide & Deep / DLRM-lite

Wide part: the hashed sparse features straight into a linear unit (memorisation).
Deep part: embeddings → MLP (generalisation).
DLRM's variation: explicit pairwise dot products between embeddings before the MLP.

Build one model with a flag that switches the interaction style, so the comparison is
apples to apples.

In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, cardinalities, dim=16, mlp=(256, 128, 64), interaction="concat"):
        super().__init__()
        # interaction: "concat" (Wide&Deep) | "dot" (DLRM)
        # TODO
        raise NotImplementedError

## 4. Training loop

Things that will bite you, in the order they usually do:

- **Class imbalance.** At a sub-1% base rate, a network happily learns to predict the
  base rate and stop. Negative downsampling fixes training speed but *breaks
  calibration* — and un-breaking it is exactly Week 3's `SamplingRateCorrector`. If you
  downsample, record the rate.
- **Embedding LR.** Sparse embedding tables usually want a much higher learning rate
  than the dense MLP. One global LR is the most common reason a DLRM underperforms.
- **Early stopping on val log-loss, not AUC.** You are going to calibrate this thing.

In [ ]:
def train_epoch(model, loader, opt, loss_fn):
    # TODO
    raise NotImplementedError

## 5. The scaling curve

Train every model at several training-set sizes and plot AUC vs rows. This is the plot
that answers the week's question.

In [ ]:
# fractions = [0.01, 0.1, 0.5, 1.0]
# rows: model x fraction -> auc, then plot
# fig, ax = plt.subplots(); ...
# print(plots.save(fig, 2, "auc_vs_training_size"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=2,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=2))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week02_* results/
git commit -m "week 02: <the finding, not the task>"
```